# Multivariate EDA

In this notebook I will perform multivariate analysis to see if there are more hidden relationships.

Questions to answer:
1. Is avg_vtat missingness confounded by combinations of features, or purely driven by the target?
2. Which feature families are redundant and which representative should be kept?
3. Does the pickup × drop interaction add signal beyond individual locations?

## Setup & Data Loading

In [ ]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif

sys.path.insert(0, os.path.abspath("../src"))
from eda_utils import cramers_v

pd.set_option("display.max_columns", 40)

In [ ]:
df = pd.read_parquet("../data/silver/data_to_multivar.parquet", engine='pyarrow')
df = df.sort_values(["date", "time"]).reset_index(drop=True)
df.head()

,date,time,vehicle_type,pickup_location,drop_location,avg_vtat,is_cancelled,vtat_zone,is_instant_arrival,is_timeout,is_long_wait,vtat_missing,hour,weekday,month,day_of_month,week_of_year,quarter,is_night,is_business,is_rush
0,2024-01-01,00:19:34,Bike,Udyog Vihar,Ambience Mall,10.8,0,normal,0,0,0,0,0,0,1,1,1,1,1,0,0
1,2024-01-01,01:35:18,Go Mini,Basai Dhankot,Madipur,8.5,0,normal,0,0,0,0,1,0,1,1,1,1,1,0,0
2,2024-01-01,01:37:50,Go Sedan,Tughlakabad,Greater Kailash,7.4,1,normal,0,0,0,0,1,0,1,1,1,1,1,0,0
3,2024-01-01,01:48:03,Auto,Palam Vihar,Kherki Daula Toll,5.6,1,normal,0,0,0,0,1,0,1,1,1,1,1,0,0
4,2024-01-01,01:49:56,Go Sedan,Narsinghpur,Pulbangash,6.2,1,normal,0,0,0,0,1,0,1,1,1,1,1,0,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 21 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   date                150000 non-null  datetime64[ns]
 1   time                150000 non-null  object        
 2   vehicle_type        150000 non-null  category      
 3   pickup_location     150000 non-null  category      
 4   drop_location       150000 non-null  category      
 5   avg_vtat            139500 non-null  float32       
 6   is_cancelled        150000 non-null  int64         
 7   vtat_zone           139500 non-null  category      
 8   is_instant_arrival  150000 non-null  Int8          
 9   is_timeout          150000 non-null  Int8          
 10  is_long_wait        150000 non-null  Int8          
 11  vtat_missing        150000 non-null  int64         
 12  hour                150000 non-null  int32         
 13  weekday             150000 no

## Multivariate Missingness Confounding Check

The bivariate analysis notebook showed that avg_vtat missingness is spread roughly equally across all independent features and I left it here to check whether combinations of these independent + dependent features can predict missingness. For that I decided to go with a log regression and I will compare it with the baseline! 

Yhe question here would be: can the combination of all non-VTAT features predict whether avg_vtat (imbalanced) is missing any better than is_cancelled alone? For that, AUC (robust to class imbalance) measures the model's overall ranking ability across all possible decision thresholds.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

non_vtat_cols = [
    "weekday", "month", "day_of_month", "week_of_year", "quarter",
    "hour", "is_night", "is_business", "is_rush",
    "vehicle_type", "pickup_location", "drop_location",
    "is_cancelled",
]

X = df[non_vtat_cols].copy()
for c in X.select_dtypes(include="category").columns:
    X[c] = LabelEncoder().fit_transform(X[c])
y = df["vtat_missing"]

lr_full = LogisticRegression(max_iter=2000, solver="lbfgs")
lr_full.fit(X, y)
y_prob_full = lr_full.predict_proba(X)[:, 1]
auc_full = roc_auc_score(y, y_prob_full)

X_base = df[["is_cancelled"]]
lr_base = LogisticRegression(max_iter=2000, solver="lbfgs")
lr_base.fit(X_base, y)
y_prob_base = lr_base.predict_proba(X_base)[:, 1]
auc_base = roc_auc_score(y, y_prob_base)

print(f"Baseline model (is_cancelled only)  AUC = {auc_base:.4f}")
print(f"Full model (all non-VTAT features)  AUC = {auc_full:.4f}")
print(f"AUC lift from adding all features   ΔAU = {auc_full - auc_base:.4f}")

coef_df = (
    pd.DataFrame({"feature": non_vtat_cols, "coef": lr_full.coef_[0]})
    .assign(abs_coef=lambda d: d["coef"].abs())
    .sort_values("abs_coef", ascending=False)
    .reset_index(drop=True)
)
print("\nLogistic regression coefficients (full model):")
coef_df

Baseline model (is_cancelled only)  AUC = 0.8656
Full model (all non-VTAT features)  AUC = 0.8674
AUC lift from adding all features   ΔAU = 0.0019

Logistic regression coefficients (full model):


,feature,coef,abs_coef
0,is_cancelled,7.199288,7.199288
1,is_business,-0.032665,0.032665
2,quarter,-0.028727,0.028727
3,is_night,-0.027788,0.027788
4,is_rush,0.020427,0.020427
5,month,-0.009505,0.009505
6,hour,-0.004837,0.004837
7,week_of_year,0.004227,0.004227
8,weekday,0.004040,0.004040
9,day_of_month,0.000604,0.000604


The full model and baseline model achieve almost the same result, so no combination of features predicts missingness beyond is_cancelled alone. The missingness mechanism is MNAR driven by the target with no hidden confounding from feature interactions.

That said, vtat_missing is not simply is_cancelled in disguise. Many cancelled rides do have a VTAT value recorded (the vehicle arrived, the rider still cancelled), and some non-cancelled rides have VTAT missing for unknown reasons, likely logging failures or system issues? 

Imputation of avg_vtat or using vtat_missing as a feature won't introduce bias from confounding with other features.

Since the dataset contains different original features and their engineered derivated, I'm going to group them in blocks and check whixh has the highest effect size over the target

In [ ]:
feature_block = {
    "temporal_date": ["weekday", "month", "day_of_month", "week_of_year", "quarter"],
    "temporal_time": ["hour", "is_rush", "is_night", "is_business"],
    "pickup_location": ["pickup_location"],
    "drop_location": ["drop_location"],
    "vehicle": ["vehicle_type"],
    "vtat": ["avg_vtat", "vtat_zone", "is_instant_arrival" ,"is_timeout", "is_long_wait", "vtat_missing"], 
    "is_cancelled": ["is_cancelled"]
}

rows = []
for block, cols in feature_block.items():
    for col in cols:
        if col == "is_cancelled":
            continue
        rates = df.groupby(col)['is_cancelled'].mean()
        rows.append({
            "block": block,
            "feature": col,
            "rate_swing": rates.max() - rates.min(),
        })

rate_swing_df = pd.DataFrame(rows).sort_values(["block", "rate_swing"], ascending=False).reset_index(drop=True)
rate_swing_df


/tmp/ipykernel_12107/2652892690.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  rates = df.groupby(col)['is_cancelled'].mean()


,block,feature,rate_swing
0,vtat,avg_vtat,1.000000
1,vtat,vtat_zone,1.000000
2,vtat,vtat_missing,0.731183
3,vtat,is_timeout,0.696346
4,vtat,is_instant_arrival,0.337299
5,vtat,is_long_wait,0.135741
6,vehicle,vehicle_type,0.009504
7,temporal_time,hour,0.032106
8,temporal_time,is_night,0.004227
9,temporal_time,is_rush,0.004124


## Family Redundancy Check

Before doing any further multivariate work I need to quantify how much overlap there is and whether keeping all three adds signal or just noise.

temporal_tim: is_night, is_rush, is_business are deterministic binary flags derived from hour. Redundant by construction. I will pick hour as representative.

temporal_date: same situation as with temporal_time variables. All of them come from date, and it was proven in the previous notebook that is carried no signal. 

vehicle, pickup_location, drop_location are all single variable each. Nothing to check.

vtat family will be checked:

#### avg_vtat Family

In [ ]:
complete = df.dropna(subset=["avg_vtat"]).copy()

groups = [g["avg_vtat"].values for _, g in complete.groupby("vtat_zone", observed=True)]
ss_bw = sum(len(g) * (g.mean() - complete["avg_vtat"].mean()) ** 2 for g in groups)
ss_tot = ((complete["avg_vtat"] - complete["avg_vtat"].mean()) ** 2).sum()
eta_sq_1 = ss_bw / ss_tot

zone_codes = complete["vtat_zone"].cat.codes.values.reshape(-1, 1).astype(float)
mi_1 = mutual_info_classif(
    complete[["avg_vtat"]].values,
    zone_codes.ravel().astype(int),
    discrete_features=[False], random_state=42, n_neighbors=5,
)[0]

print("avg_vtat - vtat_zone")
print(f"  eta-sq = {eta_sq_1:.4f}")
print(f"  MI     = {mi_1:.4f}")


cv_3 = cramers_v(df["vtat_zone"].astype(str), df["vtat_missing"].astype(str))

print("\nvtat_zone - vtat_missing")
print(f"  Cramer's V = {cv_3:.4f}")





avg_vtat - vtat_zone
  eta-sq = 0.8239
  MI     = 1.1667

vtat_zone - vtat_missing
  Cramer's V = 1.0000


vtat_zone is highly redundant (compare with avg_vtat in rate_swing_df above) so I will drop it and keep avg_vtat because it has smaller granularity, avg_vtat and vtat_missing are structurally complementary so I will keep both for linear models and (still to decide) eliminate vtat_missing while imputing sentinel value (-1) for trees

Also avg_vtat violates the log-odds linearity assumption required for logistic regression (non-monotonic relationship). Therefore Logistic Regression should use vtat_zone because it's categorical and the tree Models could use avg_vtat bc trees handle non-linearity already

Both features are kept in the working dataset for model-specific usage

In [ ]:
vars = df[["avg_vtat", "vtat_zone", "vehicle_type", "pickup_location", "drop_location", "vtat_missing", "is_cancelled"]].copy()
vars

,avg_vtat,vtat_zone,vehicle_type,pickup_location,drop_location,vtat_missing,is_cancelled
0,10.8,normal,Bike,Udyog Vihar,Ambience Mall,0,0
1,8.5,normal,Go Mini,Basai Dhankot,Madipur,0,0
2,7.4,normal,Go Sedan,Tughlakabad,Greater Kailash,0,1
3,5.6,normal,Auto,Palam Vihar,Kherki Daula Toll,0,1
4,6.2,normal,Go Sedan,Narsinghpur,Pulbangash,0,1
...,...,...,...,...,...,...,...
149995,6.9,normal,Bike,DLF Phase 3,Okhla,0,0
149996,7.9,normal,eBike,Saket,Noida Sector 62,0,0
149997,9.9,normal,Go Mini,GTB Nagar,Anand Vihar ISBT,0,0
149998,2.6,instant,Uber XL,Ashram,Vasant Kunj,0,0


## Interaction Effects: pickup × drop

Both locations had meaningful individual rate swings in the bivariate analysis. The question here is whether specific routes carry signal 

In [ ]:
route = vars.copy()
route["route"] = route["pickup_location"].astype(str) + "_" + route["drop_location"].astype(str)

target = route["is_cancelled"].astype(str)

v_pickup = cramers_v(route["pickup_location"].astype(str), target)
v_drop   = cramers_v(route["drop_location"].astype(str), target)
v_route  = cramers_v(route["route"], target)

print(f"Cramer's V with target:")
print(f"  pickup_location: {v_pickup:.4f}")
print(f"  drop_location:   {v_drop:.4f}")
print(f"  route (pick_drop): {v_route:.4f}")


Cramer's V with target:
  pickup_location: 0.0369
  drop_location:   0.0367
  route (pick_drop): 0.4485


The route combination is a far stronger than either location alone! 

Well, Cramer's V is probably inflate by the high cardinality of route but still the jump is significant so the next step would be to implement a target encoding validation with CV and then compare three versions: model with raw routes, model with target-encoded routes and model without any route. After comparing performance ROC-AUC I will know if this high value on Cramer's means real predictive power or overfitting in rare categories..

Conclusions and the final feature set for modeling are documented in the section 4 of PROJECT_WALKTROUGH.md